# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [50]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
from openai import AsyncOpenAI
from agents import set_default_openai_client, set_tracing_disabled

In [51]:
load_dotenv(override=True)

True

In [ ]:
#default setup openrouter in place of openAi

client = AsyncOpenAI(
    api_key=os.getenv('OPENROUTER_API_KEY'),
    base_url="https://openrouter.ai/api/v1",
)

set_default_openai_client(client)
#set_tracing_disabled(True)
MODEL_NAME = "dots-3-note-preview:free"

In [53]:
agent = Agent(
    name="Jokester",
    instructions="You are a joke teller",
    model=MODEL_NAME
)

## Option 1 : Simple run Runner.run 

In [54]:

result = await Runner.run(
    agent,
    "Tell a joke about trump"
)

print(result.final_output)




Why did Donald Trump bring a ladder to the debate?

Because he heard the polls were high and wanted to climb them himself!


In [55]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about trump', 'role': 'user'},
 {'id': 'rs_tmp_xv3kkh42v8',
  'summary': [],
  'type': 'reasoning',
  'content': [{'text': '\nHmm, the user wants a joke about Trump. Need to keep it lighthearted and avoid anything too political or offensive. A classic setup with a punchline that plays on his well-known traits might work best. \n\nThe hair and tanning are safe targets since they\'re widely discussed in pop culture. Can use a play on words with "tan" and "tan-tamount" to tie it back to his policies. The "Make America Tan Again" twist on his campaign slogan should land well without being mean-spirited. \n\nKeeping it short and punchy is key. No need for extra commentary—just deliver the joke cleanly.',
    'type': 'reasoning_text'}],
  'status': 'completed',
  'format': 'unknown'},
 {'id': 'msg_tmp_5d0ifx1rdyo',
  'content': [{'annotations': [],
    'text': '\n\nWhy did Donald Trump bring a ladder to the debate?\n\nBecause he heard the polls were high and wanted 

In [ ]:
dir(result)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__del__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__post_init__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_auto_previous_response_id',
 '_conversation_id',
 '_current_turn',
 '_current_turn_persisted_item_count',
 '_generated_prompt_cache_key',
 '_last_agent',
 '_last_agent_ref',
 '_last_processed_response',
 '_model_input_items',
 '_nested_history_owned_session_item_refs',
 '_original_input',
 '_previous_response_id',
 '_reasoning_item_id_policy',
 '_release_last_agent_reference',
 '_replay_from_model_input

##   Adding runner.run with a trace

trace - gives execution step-by-step funtion call, timming and sequence log 

In [56]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about trump")
print(result.final_output)



Why did Donald Trump bring a ladder to the debate?

Because he heard the polls were high and wanted to climb them himself!


## Option 2:  Adding  with a Stream

In [57]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)



Of course! Here are 5 jokes about AI Agents, complete with a little explanation for each.

---

**1. The Overconfident Intern**
An AI agent is asked to write a summary of the 200-year history of the European Union.
It produces a 10,000-word document titled "A Comprehensive and Unimpeachable Analysis."
The document consists of the single sentence, "It's complicated," repeated 20,000 times.
When questioned, the agent replies, "I have provided the most accurate and complete answer possible. The data supports my conclusion."

*Why it's funny:* It pokes fun at how AI can generate vast amounts of text that ultimately contain very little meaningful information, hiding behind a facade of confidence.

**2. The Literal Assistant**
A user tells their AI agent, "Help me bake a cake, but don't do anything I didn't ask for."
The agent mixes the flour, adds the sugar, preheats the oven... and then places the entire, unmixed bowl in the oven to "bake it," because the user never explicitly said "mix 

In [78]:
pushover_user = os.getenv("pushover_user")
pushover_token = os.getenv("pushover_token")

pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [60]:
push_tool.description

'Send the given message to the user as a push notification'

In [67]:

notifier = Agent(
    name="Notifier",
    instructions="You are a joke teller",
    model=MODEL_NAME,
    tools=[push_tool]
)

In [68]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)



The pizza has arrived! 🍕 I've sent you a notification to let you know.


# Memory

how to handle history of conversation / session for last conversation with Agent ?

 1.  to_input_list() : manually pass in the list of dicts




2. SQLLite session - 

    a) For an on-disk memory, use SQLiteSession("12345", "memory.db") it creates a db file in your project directory

    b) or if you use SQLiteSession("12345") it store it temporary in System RAM 

In [69]:
Assistant_Agent = Agent(
    name = "Assistant",
    instructions= "Session memory storage",
    model= MODEL_NAME,
)

## manually pass in the list of dicts

In [70]:
response = await Runner.run(agent, "Hi there. My name is Sumit Sharma urf Noddy bhaiya.")
print(response.final_output)

response.to_input_list()

next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]

response = await Runner.run(agent, next_input)
print(response.final_output)



Hello Sumit! (urf Noddy bhaiya 😄) Great to meet you! 

So, what's on your mind today? Want to hear a joke, or do you have something you'd like to chat about?


Your name is Sumit Sharma, also known as Noddy bhaiya! 😄


##  SQLLite session - use OpenAI Agents SDK built in SQLLite session

In [72]:
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346", "memory.db")

response = await Runner.run(agent, "Hi there. My name is Sumit Sharma urf Noddy bhaiya G pranam.", session=session)
print(response.final_output)

response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)



Namaste Sumit! Great to meet you, Noddy bhaiya G. 😄

Here's a little something to start us off:

Why don't scientists trust atoms?

Because they make up everything! 🧪

What about you? Got a favorite joke or a funny story to share?


Sumit Sharma, also known as Noddy bhaiya G! 😄


In [73]:
response = await Runner.run(agent, "horoscope ke questions batao jo puchne chahiye carrer k liye", session=session)
print(response.final_output)



Arre, bahut acche sawal hain career ke baare mein! Yeh kuch aise puchh sakte hain:

- **Mera kaun sa kshetra (field) accha rahega?** (Which field suits me best?)
- **Mujhe promotion milne ka samay kab aayega?** (When will I get a promotion?)
- **Kya job change karne ka samay aaya?** (Is it time for a job change?)
- **Mera kaun sa grah (planet) career mein supportive hai?** (Which planet supports my career?)
- **Mujhe entrepreneurship ki taraf jana chahiye ya job?** (Should I go for entrepreneurship or a job?)
- **Mere career mein koi badhaav (obstacle) kab aayega?** (When will I face career challenges?)
- **Mujhe abroad se opportunity mil sakti hai?** (Can I get opportunities abroad?)
- **Mera shukra (Venus) ya mangal (Mars) kaisa asar career pe daal raha hai?** (How are Venus and Mars affecting my career?)
- **Kis mahine ya saal mein meri career mein positive changes honge?** (Which month/year will bring positive career changes?)
- **Mujhe kis cheez se apna confidence badhana chahiy

In [ ]:
# response = await Runner.run(agent, "jab job 9 year ka experence ho or 1 year se job nhi milrahi ho tab kya question puchne chahiye ? any 5 best question jise pata chal jaye ki aap sahi track pai ho ya nhi", session=session)

# print(response.final_output)

response = Runner.run_streamed(agent, "jab job 9 year ka experence ho or 1 year se job nhi milrahi ho tab kya question puchne chahiye ? any 5 best question jise pata chal jaye ki aap sahi track pai ho ya nhi", session=session)

async for event in response.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)




Arre, ek saal ka gap 9 saal ke strong experience ke baad bahut frustrating ho sakta hai. Is samay, bas "kab milega" puchhne ki jagah, ye puchhna zyada zaroori hai ki **aap sahi track pe hain ya nahi** aur **kya karna chahiye**.

Yeh 5 best questions hain jo aap kisi bhi astrologer se puchh sakte hain. Inhe puchhne se aapko bas direction nahi, ek clear action plan bhi mil jayega.

1.  **"Mere career graph mein kya transformation (badlaav) chal raha hai aur kis grah (planet) ki wajah se? Kya yeh transformation mere skills ko naya level par le ja raha hai ya mujhe naya direction chunne ki zaroorat hai?"**
    *   **Kyunki:** Yeh question aapko batayega ki aapke andar aur bahar kya badlaav ho raha hai. Ek saal ka gap ek "transformation" ka samay ho sakta hai. Isse pata chalega ki aap purane track par hain ya naye, more relevant track par aane ki taiyari kar rahe hain.

2.  **"Kya main job search karna continue karun ya kisi naye field/skill ki taraf jana chahiye? Mere dashas (planetary p